# 📖 Notebook 3: Real-Time Bid Notifications

When someone places a bid on an auction you're watching, you want to know **immediately** — not 30 seconds later. Stale bid data leads to frustrated users who bid based on outdated information and get rejected.

In this notebook, we'll build progressively better notification systems:
1. **Polling** — the simplest (but most wasteful) approach
2. **Redis Pub/Sub** — push notifications to all watchers instantly
3. **Full simulation** — multiple users watching an auction and receiving live updates

## Learning Objectives

By the end of this notebook, you'll understand:
- Why polling wastes resources and delivers stale data
- How Redis Pub/Sub enables real-time push notifications
- How Server-Sent Events (SSE) would work in a real web app
- The scaling challenge: coordinating updates across multiple servers

## 🛠️ Setup

Start the infrastructure first:

```bash
cd system-designs/online-auction
docker-compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `auction_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import redis
import time
import json
import threading
from datetime import datetime

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "auction_demo",
    "user": "demo",
    "password": "demo"
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_db_connection():
    return psycopg2.connect(**DB_CONFIG)

def get_redis_client():
    return redis.Redis(**REDIS_CONFIG)

# Test connections
try:
    conn = get_db_connection()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

try:
    r = get_redis_client()
    r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")

---
## 🔄 Approach 1: Polling

The simplest way to keep bid data fresh: **ask the database every few seconds**.

```
Client: "What's the max bid?"  → Server: "$1,000"
  ... 3 seconds later ...
Client: "What's the max bid?"  → Server: "$1,000"  (no change)
  ... 3 seconds later ...
Client: "What's the max bid?"  → Server: "$1,200"  (new bid!)
  ... 3 seconds later ...
Client: "What's the max bid?"  → Server: "$1,200"  (no change again)
```

### Problems
1. **Wasteful** — most polls return the same data (no new bids)
2. **Delayed** — up to 3 seconds of stale data
3. **Doesn't scale** — 10,000 users watching = 10,000 DB queries every 3 seconds

In [ ]:
def poll_max_bid(auction_id):
    """Simulate one poll: query the DB for the current max bid."""
    conn = get_db_connection()
    cur = conn.cursor()
    cur.execute(
        "SELECT max_bid_amount, max_bid_user_id FROM auctions WHERE id = %s",
        (auction_id,)
    )
    row = cur.fetchone()
    conn.close()
    return {"max_bid": float(row[0]), "leader": row[1]}

# Simulate 10 watchers polling every 2 seconds for 10 seconds
AUCTION_ID = 1
NUM_WATCHERS = 10
POLL_INTERVAL = 2
POLL_DURATION = 10

print(f"📊 Simulating {NUM_WATCHERS} users polling auction #{AUCTION_ID}")
print(f"   Poll interval: every {POLL_INTERVAL}s for {POLL_DURATION}s\n")

total_queries = 0
useful_queries = 0
last_bid = None

start = time.time()
while time.time() - start < POLL_DURATION:
    # Each watcher polls independently
    for watcher_id in range(NUM_WATCHERS):
        result = poll_max_bid(AUCTION_ID)
        total_queries += 1
        if result["max_bid"] != last_bid:
            useful_queries += 1
            last_bid = result["max_bid"]

    elapsed = time.time() - start
    print(f"   [{elapsed:.0f}s] Max bid: ${last_bid:,.2f} — {NUM_WATCHERS} queries fired")
    time.sleep(POLL_INTERVAL)

wasted = total_queries - useful_queries
waste_pct = (wasted / total_queries) * 100 if total_queries > 0 else 0

print(f"\n📊 Polling stats:")
print(f"   Total DB queries:  {total_queries}")
print(f"   Useful (new data): {useful_queries}")
print(f"   Wasted (no change):{wasted}")
print(f"   Waste rate:        {waste_pct:.0f}%")
print(f"\n💡 In a real system with millions of watchers, this would crush the database.")

---
## 📡 Approach 2: Redis Pub/Sub

Instead of watchers asking "is there a new bid?" every few seconds, we **push** the update to them the moment it happens.

### How Redis Pub/Sub Works

Redis has a built-in **publish/subscribe** system:

```
  Publisher                    Redis                   Subscribers
  (Bid Service)               (Channel)               (Watchers)
                                                                   
  PUBLISH ──────────►  auction:1:bids  ──────────►  User A  
  "new bid: $1,200"                    ──────────►  User B
                                       ──────────►  User C
```

- **Publisher**: When a bid is accepted, publish a message to `auction:{id}:bids`
- **Subscriber**: Each watcher subscribes to the channel for their auction
- **Delivery**: Redis pushes the message to ALL subscribers instantly

**Key advantage**: Zero wasted queries. Messages are sent only when something actually changes.

In [ ]:
# First, let's see basic Redis Pub/Sub in action

r = get_redis_client()

# Messages received by our subscriber
received_messages = []

def subscriber_thread(channel, subscriber_name, duration=5):
    """
    Subscribe to a Redis channel and collect messages.
    Each subscriber needs its own Redis connection.
    """
    sub_client = get_redis_client()
    pubsub = sub_client.pubsub()
    pubsub.subscribe(channel)

    start = time.time()
    for message in pubsub.listen():
        if time.time() - start > duration:
            break
        if message["type"] == "message":
            data = json.loads(message["data"])
            received_messages.append({
                "subscriber": subscriber_name,
                "data": data,
                "received_at": datetime.now().strftime("%H:%M:%S.%f")[:-3]
            })

    pubsub.unsubscribe()
    pubsub.close()

# Start 3 subscribers watching the same auction channel
channel = "auction:1:bids"
threads = []
for name in ["Alice", "Bob", "Charlie"]:
    t = threading.Thread(target=subscriber_thread, args=(channel, name, 5))
    t.start()
    threads.append(t)

# Give subscribers time to connect
time.sleep(0.5)

# Publish 3 bid updates
print(f"📡 Publishing bid updates to channel '{channel}'...\n")

bids = [
    {"user_id": 10, "amount": 9000.00, "timestamp": datetime.now().isoformat()},
    {"user_id": 20, "amount": 9500.00, "timestamp": datetime.now().isoformat()},
    {"user_id": 30, "amount": 10000.00, "timestamp": datetime.now().isoformat()},
]

for bid in bids:
    r.publish(channel, json.dumps(bid))
    print(f"   📤 Published: User {bid['user_id']} bid ${bid['amount']:,.2f}")
    time.sleep(0.3)

# Wait for subscribers to receive messages
for t in threads:
    t.join()

print(f"\n📥 Messages received ({len(received_messages)} total):\n")
for msg in received_messages:
    print(f"   {msg['subscriber']:>8} received: User {msg['data']['user_id']} bid ${msg['data']['amount']:,.2f} at {msg['received_at']}")

print(f"\n💡 All 3 subscribers got all 3 messages — instantly, with zero polling!")

---
## 🏗️ Building a Notification System

Now let's combine everything: bids go through our concurrency-safe bid processor, and bid updates are pushed to watchers via Pub/Sub.

### Architecture
```
  User places bid
       │
       ▼
  ┌─────────────┐     ┌───────────┐
  │ Bid Service  │────►│ PostgreSQL │  (permanent storage)
  └──────┬───────┘     └───────────┘
         │
         │ PUBLISH
         ▼
  ┌─────────────┐
  │    Redis     │  (Pub/Sub channel)
  └──────┬───────┘
         │
    ┌────┼────┐
    ▼    ▼    ▼
  User  User  User  (watchers)
```

In [ ]:
def place_bid_with_notification(auction_id, user_id, amount):
    """
    Place a bid using OCC (from Notebook 1), then publish
    a notification via Redis Pub/Sub if accepted.
    """
    conn = get_db_connection()
    cur = conn.cursor()

    try:
        # OCC: Read current max
        cur.execute("SELECT max_bid_amount FROM auctions WHERE id = %s", (auction_id,))
        current_max = float(cur.fetchone()[0])

        if amount <= current_max:
            cur.execute(
                "INSERT INTO bids (auction_id, user_id, amount, status) VALUES (%s, %s, %s, 'rejected')",
                (auction_id, user_id, amount)
            )
            conn.commit()
            conn.close()
            return {"status": "rejected", "amount": amount}

        # OCC: Try to update (only if max hasn't changed)
        cur.execute(
            """UPDATE auctions SET max_bid_amount = %s, max_bid_user_id = %s
               WHERE id = %s AND max_bid_amount = %s""",
            (amount, user_id, auction_id, current_max)
        )

        if cur.rowcount == 1:
            cur.execute(
                "INSERT INTO bids (auction_id, user_id, amount, status) VALUES (%s, %s, %s, 'accepted')",
                (auction_id, user_id, amount)
            )
            conn.commit()
            conn.close()

            # 🔔 Publish notification to all watchers
            r = get_redis_client()
            notification = json.dumps({
                "auction_id": auction_id,
                "new_max_bid": amount,
                "bidder_id": user_id,
                "timestamp": datetime.now().isoformat()
            })
            num_subscribers = r.publish(f"auction:{auction_id}:bids", notification)

            return {
                "status": "accepted",
                "amount": amount,
                "notified_subscribers": num_subscribers
            }
        else:
            conn.rollback()
            conn.close()
            return {"status": "conflict", "amount": amount}

    except Exception as e:
        conn.rollback()
        conn.close()
        return {"status": "error", "error": str(e)}

In [ ]:
# Full simulation: watchers subscribe, then bids come in

AUCTION_ID = 5  # MacBook Pro auction
received = []

def watcher(name, auction_id, duration=8):
    """Simulate a user watching an auction for real-time bid updates."""
    client = get_redis_client()
    pubsub = client.pubsub()
    pubsub.subscribe(f"auction:{auction_id}:bids")

    start = time.time()
    for message in pubsub.listen():
        if time.time() - start > duration:
            break
        if message["type"] == "message":
            data = json.loads(message["data"])
            received.append({"watcher": name, "data": data})

    pubsub.unsubscribe()
    pubsub.close()

# Start 5 watchers
print(f"👀 Starting 5 watchers on auction #{AUCTION_ID}...\n")
watcher_names = ["Alice", "Bob", "Charlie", "Diana", "Eve"]
watcher_threads = []
for name in watcher_names:
    t = threading.Thread(target=watcher, args=(name, AUCTION_ID, 8))
    t.start()
    watcher_threads.append(t)

time.sleep(0.5)  # let subscribers connect

# Now simulate a bidding war
print("⚡ Bidding war begins!\n")

# Read current max so we can bid above it
conn = get_db_connection()
cur = conn.cursor()
cur.execute("SELECT max_bid_amount FROM auctions WHERE id = %s", (AUCTION_ID,))
current = float(cur.fetchone()[0])
conn.close()

bidders = [
    (31, current + 100),
    (32, current + 300),
    (33, current + 500),
    (34, current + 800),
]

for user_id, amount in bidders:
    result = place_bid_with_notification(AUCTION_ID, user_id, amount)
    emoji = "✅" if result["status"] == "accepted" else "❌"
    subs = result.get("notified_subscribers", 0)
    print(f"   {emoji} User {user_id} bid ${amount:,.2f} → {result['status']} (notified {subs} watchers)")
    time.sleep(0.5)

# Wait for watchers to finish
for t in watcher_threads:
    t.join()

print(f"\n📥 Notifications received by watchers ({len(received)} total):\n")

# Group by watcher
by_watcher = {}
for msg in received:
    name = msg["watcher"]
    if name not in by_watcher:
        by_watcher[name] = []
    by_watcher[name].append(msg["data"])

for name in watcher_names:
    msgs = by_watcher.get(name, [])
    print(f"   {name}: received {len(msgs)} updates")
    for m in msgs:
        print(f"      → New max bid: ${m['new_max_bid']:,.2f} by User {m['bidder_id']}")

print(f"\n🎉 All watchers received all bid updates in real-time!")

---
## 📊 Polling vs Pub/Sub: Resource Comparison

Let's compare the resource usage of both approaches with some realistic numbers.

In [ ]:
# Resource comparison: Polling vs Pub/Sub

print("📊 Resource Comparison: Polling vs Pub/Sub")
print("=" * 65)
print()

# Scenario: 1 auction, 100 bids over 1 hour, 10,000 watchers
watchers = 10_000
bids_per_hour = 100
poll_interval_sec = 3
duration_hours = 1

# Polling: every watcher queries every N seconds
polls_per_watcher = (duration_hours * 3600) / poll_interval_sec
total_polls = watchers * polls_per_watcher
useful_polls = bids_per_hour * watchers  # only these return new data
wasted_polls = total_polls - useful_polls

# Pub/Sub: only send messages when bids happen
pubsub_messages = bids_per_hour * watchers  # one message per watcher per bid

print(f"Scenario: {watchers:,} watchers, {bids_per_hour} bids/hour, 1 hour")
print()
print(f"{'Metric':<35} {'Polling':>15} {'Pub/Sub':>15}")
print("-" * 65)
print(f"{'Total messages/queries':<35} {total_polls:>15,.0f} {pubsub_messages:>15,}")
print(f"{'Useful (contain new data)':<35} {useful_polls:>15,} {pubsub_messages:>15,}")
print(f"{'Wasted (no change)':<35} {wasted_polls:>15,.0f} {'0':>15}")
print(f"{'Waste percentage':<35} {wasted_polls/total_polls*100:>14.1f}% {'0.0%':>15}")
print(f"{'DB load':<35} {'Very High':>15} {'Zero':>15}")
print(f"{'Max staleness':<35} {f'{poll_interval_sec}s':>15} {'~0ms':>15}")

print()
print(f"💡 Pub/Sub eliminates {wasted_polls:,.0f} wasted queries ({wasted_polls/total_polls*100:.1f}% of all polling traffic).")
print(f"   With Pub/Sub, your database is completely untouched by watchers.")

---
## 🌐 How This Works in a Real Web App: Server-Sent Events (SSE)

In a real web application, you'd use **Server-Sent Events (SSE)** to push updates from the server to the browser. Here's how the pieces fit together:

```
  Browser                  Server                  Redis
  ───────                  ──────                  ─────
  GET /auctions/1/stream
  ─────────────────────►
                           SUBSCRIBE auction:1:bids
                           ────────────────────────►
                           
  (connection stays open)
                           
  ... someone places a bid ...
                           
                           ◄──── message: {bid data}
  ◄── data: {bid data}
  
  updateUI(newMaxBid)
```

The browser opens a single long-lived connection. The server subscribes to Redis and forwards messages as SSE events. No polling needed.

### Client-Side Code (JavaScript)
```javascript
const eventSource = new EventSource('/api/auctions/1/bid-stream');

eventSource.onmessage = (event) => {
    const { new_max_bid, bidder_id } = JSON.parse(event.data);
    document.getElementById('max-bid').textContent = `$${new_max_bid}`;
};
```

### Why SSE Over WebSockets?
- SSE is **unidirectional** (server → client) — perfect for bid updates
- SSE is simpler to implement (just HTTP with `text/event-stream`)
- Built-in automatic reconnection
- WebSockets would be needed if clients also send data (e.g., chat)

---
## 🔧 Advanced: Simulating Multi-Server Coordination

In production, you'll have **multiple servers** handling SSE connections. When a bid arrives at Server A, users connected to Server B also need the update.

Redis Pub/Sub solves this: every server subscribes to the same channel. When one server publishes a bid update, all servers receive it and forward to their connected clients.

Let's simulate this with threads representing different servers.

In [ ]:
# Simulate 3 servers, each with their own SSE connections

server_logs = {"Server-A": [], "Server-B": [], "Server-C": []}

def simulated_server(server_name, auction_id, connected_users, duration=8):
    """
    Simulate a server that:
    1. Subscribes to Redis Pub/Sub for bid updates
    2. Forwards updates to its connected users (simulated)
    """
    client = get_redis_client()
    pubsub = client.pubsub()
    pubsub.subscribe(f"auction:{auction_id}:bids")

    start = time.time()
    for message in pubsub.listen():
        if time.time() - start > duration:
            break
        if message["type"] == "message":
            data = json.loads(message["data"])
            # Forward to all connected users on this server
            for user in connected_users:
                server_logs[server_name].append({
                    "user": user,
                    "bid": data["new_max_bid"],
                    "from_bidder": data["bidder_id"]
                })

    pubsub.unsubscribe()
    pubsub.close()

# Server A has Alice and Bob
# Server B has Charlie and Diana
# Server C has Eve
AUCTION_ID = 7  # Pokemon card auction

print("🖥️  Multi-server simulation")
print("   Server-A: Alice, Bob")
print("   Server-B: Charlie, Diana")
print("   Server-C: Eve")
print()

threads = [
    threading.Thread(target=simulated_server, args=("Server-A", AUCTION_ID, ["Alice", "Bob"], 8)),
    threading.Thread(target=simulated_server, args=("Server-B", AUCTION_ID, ["Charlie", "Diana"], 8)),
    threading.Thread(target=simulated_server, args=("Server-C", AUCTION_ID, ["Eve"], 8)),
]

for t in threads:
    t.start()

time.sleep(0.5)

# Bid arrives at "Server A" but all servers need to know
print("💥 New bid arrives (processed by Server-A, but published to all)...\n")

conn = get_db_connection()
cur = conn.cursor()
cur.execute("SELECT max_bid_amount FROM auctions WHERE id = %s", (AUCTION_ID,))
current = float(cur.fetchone()[0])
conn.close()

result = place_bid_with_notification(AUCTION_ID, user_id=50, amount=current + 500)
print(f"   Bid result: {result}")

time.sleep(1)

# Let's place another bid
result2 = place_bid_with_notification(AUCTION_ID, user_id=48, amount=current + 1000)
print(f"   Bid result: {result2}")

for t in threads:
    t.join()

print(f"\n📥 Server logs (who got notified):\n")
for server, logs in server_logs.items():
    print(f"   {server}:")
    if logs:
        for log in logs:
            print(f"      → Sent to {log['user']}: new max ${log['bid']:,.2f} (from User {log['from_bidder']})")
    else:
        print(f"      (no messages)")

print(f"\n🎉 All servers received the update via Redis Pub/Sub!")
print(f"   Even though the bid was processed by one server, every connected user got notified.")

---
## 🧠 Summary

| Approach | Latency | Resource Usage | Complexity | Best For |
|----------|---------|----------------|------------|----------|
| Polling | 0–N seconds | 🔴 Very high | Low | Prototypes, low traffic |
| Long Polling | ~0 seconds | 🟡 Medium | Medium | Moderate traffic |
| SSE + Pub/Sub | ~0ms | 🟢 Low | Medium | Production auction systems |
| WebSockets | ~0ms | 🟢 Low | High | Bidirectional (chat + bids) |

### Key Takeaways

1. **Polling is simple but wasteful** — fine for a prototype, not for production
2. **Redis Pub/Sub** provides instant delivery with zero wasted queries
3. **SSE** is the ideal browser transport for unidirectional updates (server → client)
4. **Multi-server coordination** is solved by having all servers subscribe to Redis Pub/Sub
5. **Redis Pub/Sub is fire-and-forget** — if a subscriber is offline when a message is published, they miss it. For durability, combine with a persistent store.

### Production Architecture

```
Client ──► API Gateway ──► Bid Service ──► PostgreSQL (permanent storage)
                                   │
                                   ├──► Kafka (durability, ordering)
                                   │
                                   └──► Redis Pub/Sub ──► SSE Servers ──► Browsers
```

### What You've Learned Across All 3 Notebooks

| Notebook | Core Problem | Solution |
|----------|-------------|----------|
| 1. Bid Processing | Race conditions with concurrent bids | OCC, row locking, Redis Lua |
| 2. Lifecycle | Managing auction states and expiration | State machine, idempotent closers, anti-sniping |
| 3. Notifications | Pushing bid updates to watchers | Redis Pub/Sub, SSE, multi-server coordination |